# Tugas Praktikum 3: Classification dengan Ensemble Method & Class Imbalance

**Mata Kuliah:** Data Mining  
**Nama:** Muhamad Nafi Mulyo  
**NRP:** 5054251005  
**Dataset:** Credit Card Fraud Detection (Kaggle)  

## 1. Import Library dan Load Dataset

Pertama kita import dulu semua library yang dibutuhin. Ada `pandas` dan `numpy` buat manipulasi data, `matplotlib` dan `seaborn` buat visualisasi, `scikit-learn` buat model dan evaluasi, sama `imbalanced-learn` buat SMOTE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Semua library berhasil diimport!")

: 

In [ ]:
df = pd.read_csv("creditcard.csv")

print("Shape dataset:", df.shape)
print("\n5 baris pertama:")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

Dataset punya **284.807 baris** dan **31 kolom**. Kolom V1 sampai V28 itu hasil PCA (buat jaga privasi), sedangkan `Time`, `Amount`, dan `Class` kolom aslinya. Tidak ada missing values.

## 2. Praproses Data

### 2.1 Distribusi Kelas

Kita lihat dulu seberapa parah imbalance-nya.

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Jumlah': class_counts, 'Persentase (%)': class_pct.round(4)})
dist_df.index = ['Normal (0)', 'Fraud (1)']
print(dist_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#4878CF', '#D65F5F']
bars = axes[0].bar(['Normal (0)', 'Fraud (1)'], class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Kelas Transaksi', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Transaksi')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, class_counts.max() * 1.15)

wedge_props = dict(width=0.55, edgecolor='white', linewidth=2)
axes[1].pie(
    class_counts.values,
    labels=[f'Normal (0)\n{class_pct[0]:.2f}%', f'Fraud (1)\n{class_pct[1]:.2f}%'],
    colors=colors, autopct='%1.2f%%', startangle=90,
    wedgeprops=wedge_props, textprops={'fontsize': 11}
)
axes[1].set_title('Proporsi Kelas', fontsize=14, fontweight='bold')

plt.suptitle('Class Imbalance — Credit Card Fraud Detection', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Rasio: 1 fraud per {class_counts[0]//class_counts[1]} transaksi normal")

Imbalance-nya ekstrem banget — fraud cuma **0.17%** dari total data. Kalau model asal tebak 'Normal' semua, accuracy-nya udah 99.83% tapi gak ada gunanya sama sekali.

### 2.2 Train-Test Split (80:20)

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data latih : {X_train.shape[0]:,} baris")
print(f"Data uji   : {X_test.shape[0]:,} baris")
print(f"\nKelas — Train:")
print(y_train.value_counts())
print(f"\nKelas — Test:")
print(y_test.value_counts())

### 2.3 SMOTE — Oversampling Data Latih

SMOTE (Synthetic Minority Over-sampling Technique) membuat data sintetis untuk kelas minoritas (fraud) dengan interpolasi antar sampel yang sudah ada. Ini lebih baik dari random oversampling karena data baru yang dihasilkan lebih bervariasi.

**Penting:** SMOTE cuma diterapkan ke **data latih** — data uji tetap pakai distribusi asli.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Sebelum SMOTE : {X_train.shape[0]:,} sampel")
print(f"Setelah SMOTE : {X_train_smote.shape[0]:,} sampel")
print(f"\nDistribusi kelas setelah SMOTE:")
print(pd.Series(y_train_smote).value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
before = y_train.value_counts()
after = pd.Series(y_train_smote).value_counts()

for ax, (title, data) in zip(axes, [('Sebelum SMOTE', before), ('Setelah SMOTE', after)]):
    bars = ax.bar(['Normal', 'Fraud'], data.values, color=['#4878CF', '#D65F5F'])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Jumlah Sampel')
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(data.values)*0.02,
                f'{val:,}', ha='center', fontweight='bold')

plt.suptitle('Perbandingan Distribusi Kelas Sebelum dan Sesudah SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Membangun Model Klasifikasi

Kita bandingkan 3 model Random Forest dengan perlakuan class imbalance yang berbeda:

| Model | Pendekatan |
|---|---|
| **Model 1** | RF Baseline — tanpa penanganan imbalance |
| **Model 2** | RF + SMOTE — oversampling pada data latih |
| **Model 3** | RF + `class_weight='balanced'` — cost-sensitive learning |

In [ ]:
# Model 1: Baseline
print("[1/3] Training RF Baseline...")
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)
y_pred_m1 = rf_baseline.predict(X_test)
y_prob_m1 = rf_baseline.predict_proba(X_test)[:, 1]
print("   Done!")

In [ ]:
# Model 2: RF + SMOTE
print("[2/3] Training RF + SMOTE...")
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_m2 = rf_smote.predict(X_test)
y_prob_m2 = rf_smote.predict_proba(X_test)[:, 1]
print("   Done!")

In [ ]:
# Model 3: class_weight='balanced'
print("[3/3] Training RF Balanced...")
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_balanced.fit(X_train, y_train)
y_pred_m3 = rf_balanced.predict(X_test)
y_prob_m3 = rf_balanced.predict_proba(X_test)[:, 1]
print("   Done!")
print("\nSemua model selesai ditraining!")

## 4. Evaluasi Model

### 4.1 Classification Report

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    print(f"{'='*58}")
    print(f" {name}")
    print(f"{'='*58}")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))
    print(f" AUC-ROC: {auc:.4f}\n")
    return auc

auc_m1 = evaluate_model("Model 1 — RF Baseline", y_test, y_pred_m1, y_prob_m1)
auc_m2 = evaluate_model("Model 2 — RF + SMOTE", y_test, y_pred_m2, y_prob_m2)
auc_m3 = evaluate_model("Model 3 — RF class_weight=balanced", y_test, y_pred_m3, y_prob_m3)

### 4.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, y_pred) in zip(axes, [
    ("Model 1\nRF Baseline", y_pred_m1),
    ("Model 2\nRF + SMOTE", y_pred_m2),
    ("Model 3\nRF Balanced", y_pred_m3)
]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Fraud']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrix — Perbandingan Ketiga Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for name, y_prob, auc, color in [
    ("RF Baseline", y_prob_m1, auc_m1, '#4878CF'),
    ("RF + SMOTE", y_prob_m2, auc_m2, '#D65F5F'),
    ("RF Balanced", y_prob_m3, auc_m3, '#6ACC65'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", color=color, linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier (AUC = 0.5000)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — Perbandingan Ketiga Model', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Tabel dan Grafik Perbandingan Metrik

In [ ]:
comparison = []
for name, y_pred, y_prob in [
    ("RF Baseline", y_pred_m1, y_prob_m1),
    ("RF + SMOTE", y_pred_m2, y_prob_m2),
    ("RF Balanced", y_pred_m3, y_prob_m3),
]:
    comparison.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_test, y_pred)*100:.2f}%",
        'Precision': f"{precision_score(y_test, y_pred):.4f}",
        'Recall': f"{recall_score(y_test, y_pred):.4f}",
        'F1-Score': f"{f1_score(y_test, y_pred):.4f}",
        'AUC-ROC': f"{roc_auc_score(y_test, y_prob):.4f}"
    })

comp_df = pd.DataFrame(comparison).set_index('Model')
print("Tabel Perbandingan Performa (fokus kelas Fraud):")
comp_df

In [ ]:
metrics_data = {
    'Precision (Fraud)': [precision_score(y_test, p) for p in [y_pred_m1, y_pred_m2, y_pred_m3]],
    'Recall (Fraud)': [recall_score(y_test, p) for p in [y_pred_m1, y_pred_m2, y_pred_m3]],
    'F1-Score (Fraud)': [f1_score(y_test, p) for p in [y_pred_m1, y_pred_m2, y_pred_m3]],
    'AUC-ROC': [auc_m1, auc_m2, auc_m3]
}
model_names = ['RF Baseline', 'RF + SMOTE', 'RF Balanced']
colors = ['#4878CF', '#D65F5F', '#6ACC65']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (metric, values) in zip(axes.flatten(), metrics_data.items()):
    bars = ax.bar(model_names, values, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Perbandingan Metrik Evaluasi — Kelas Fraud (1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Kenapa Accuracy Doang Gak Cukup?

Pada dataset ini, kalau model asal tebak semuanya "Normal", accuracy-nya udah **99.83%** — tapi gak bisa deteksi fraud sama sekali. Ini namanya **accuracy paradox** dan sering kejadian di dataset yang sangat imbalanced.

Metrik yang lebih relevan di sini:
- **Recall (fraud):** Seberapa banyak fraud yang berhasil ketangkep. Miss satu fraud = kerugian nyata.
- **Precision (fraud):** Seberapa sering prediksi 'fraud' beneran fraud. False alarm terus mengganggu nasabah.
- **F1-Score:** Harmonic mean precision-recall, keseimbangan keduanya.
- **AUC-ROC:** Kemampuan model membedakan dua kelas di berbagai threshold. Semakin dekat ke 1.0 makin bagus.

## 5. Kesimpulan

1. **Dataset sangat imbalanced** — fraud cuma 0.17% dari total, artinya kita butuh teknik khusus supaya model bisa mengenali pola fraud dengan benar.

2. **RF Baseline** punya akurasi tertinggi, tapi recall untuk fraud relatif rendah — banyak fraud yang lolos tidak terdeteksi.

3. **RF + SMOTE** berhasil meningkatkan recall secara signifikan. Model jadi lebih peka terhadap pola fraud karena dilatih dengan data yang lebih seimbang.

4. **RF dengan class_weight='balanced'** juga memberikan hasil kompetitif dan lebih gampang diimplementasikan karena tidak perlu resampling eksplisit.

5. Dalam konteks fraud detection, **recall adalah metrik terpenting** — lebih baik false alarm sedikit daripada fraud lolos. Kedua model dengan penanganan imbalance jelas unggul.

6. **Accuracy doang menyesatkan** untuk kasus seperti ini — selalu pakai precision, recall, F1-score, dan AUC-ROC sebagai evaluasi utama.